# Imports

In [ ]:
print("yes")

yes


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
import numpy as np
import subprocess
import pandas as pd
import numpy as np
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
#from sklearn.feature_extraction.text import CountVectorizer  #DT does not take strings as input for the model fit step....
from IPython.display import Image
#import pydotplus as pydot
from sklearn import tree
warnings.filterwarnings('ignore')


# Data Cleaning

In [ ]:
# Download latest version
# import kagglehub
# path = kagglehub.dataset_download("ericanacletoribeiro/cicids2017-cleaned-and-preprocessed")
# print("Path to dataset files:", path)
#data = pd.read_csv('/kaggle/input/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv')


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
path = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/FINAL PROJECT/cicids2017_cleaned.csv'
data = pd.read_csv(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#data.columns

In [ ]:
#print(data.info(1))
data = data.sample(frac = 0.05)
data.shape

(126038, 53)

In [ ]:
print(data['Attack Type'].unique())

['Normal Traffic' 'DoS' 'Port Scanning' 'DDoS' 'Bots' 'Web Attacks'
 'Brute Force']


In [ ]:
#data['label'].unique()
#data = pd.get_dummies(data, columns=['network_ports_all', 'network_ports_dst', 'network_ports_src'])

In [ ]:
## encoding labels
labels_dict = {
    'Normal Traffic': 0,
    'Port Scanning':1,
    'Web Attacks':2,
    'Brute Force':3,
    'DDoS':4,
    'Bots':5,
    'DoS':6
}
data['label'] = data['Attack Type'].map(labels_dict)
data = data.drop(columns='Attack Type')


In [ ]:
#data.head(1)
data.shape

(4726, 53)

# Using Decision Tree to Figure out what feature is the most important


[Website to Display the Decision Tree](http://webgraphviz.com/)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
# X = data.iloc[:,0:52].values
# Y = data['label'].values
X = data.drop("label", axis=1)
Y = data.pop("label")
x_train, x_test, y_train, y_test = train_test_split(X, Y,
                                                    test_size=0.3,
                                                    random_state=123,
                                                    shuffle=True)


# KNN Model Training
* data is 53 columns aka [2520751, 53]
* X_train = [(1764525, 52)]

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(x_train, y_train)


KNeighborsClassifier(n_neighbors=2)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Convert y_test to a binary target, assuming class 1 (Port Scanning) is the positive class.
# This aligns with `knn_model.predict_proba(x_test)[:,1]` which is the probability of class 1.
y_test_binary = (y_test == 1).astype(int)

pred = knn_model.predict_proba(x_test)[:,1]
for th in [0.1, 0.3, 0.5, 0.7, 0.9]:
    sk_hat  = (pred >= th).astype(int)

    # Use the binarized y_test for evaluation
    sk_acc  = accuracy_score(y_test_binary, sk_hat)
    sk_prec = precision_score(y_test_binary, sk_hat, zero_division=0)
    sk_rec = recall_score(y_test_binary, sk_hat)

    for metric, sk in [('Accuracy', sk_acc),
                               ('Precision', sk_prec),
                               ('Recall', sk_rec)]:
        #match = '✓' if abs(sk) < 1e-4 else '✗'
        print(f'{th:<6} {metric:<12} {sk:>10.4f}')
    print()

0.1    Accuracy         0.9873
0.1    Precision        0.8333
0.1    Recall           0.8000

0.3    Accuracy         0.9873
0.3    Precision        0.8333
0.3    Recall           0.8000

0.5    Accuracy         0.9873
0.5    Precision        0.8333
0.5    Recall           0.8000

0.7    Accuracy         0.9838
0.7    Precision        0.8649
0.7    Recall           0.6400

0.9    Accuracy         0.9838
0.9    Precision        0.8649
0.9    Recall           0.6400



In [ ]:
TSHARK_PATH = r'C:\Program Files\Wireshark\tshark.exe'

def capture_packets(interface='Wi-Fi', duration=10):
    print("[i] Capturing traffic...")
    cmd = [
        TSHARK_PATH, '-i', interface, '-T', 'fields',
        '-e', 'frame.time_epoch',
        '-e', 'ip.ttl',
        '-e', 'ip.hdr_len',
        '-e', 'ip.len',
        '-e', 'ip.flags.mf',
        '-e', 'ip.frag_offset',
        '-E', 'header=y', '-E', 'separator=,',
        '-a', f'duration:{duration}'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    from io import StringIO
    packets = pd.read_csv(StringIO(result.stdout))
    return packets

In [ ]:
def aggregate_features(packets):
    features = {}

    features['network_ttl_avg'] = packets['ip.ttl'].mean()
    features['network_ttl_max'] = packets['ip.ttl'].max()
    features['network_ttl_min'] = packets['ip.ttl'].min()
    features['network_header_length_avg'] = packets['ip.hdr_len'].mean()
    features['network_header_length_max'] = packets['ip.hdr_len'].max()
    features['network_fragmented_packets'] = (packets['ip.frag_offset'] > 0).sum()
    features['network_fragmentation_score'] = features['network_fragmented_packets'] / len(packets)
    features['log_messages_count'] = len(packets)

    return features

def predict_traffic(rf_classifier, feature_columns, interface='eth0', duration=10):
    packets = capture_packets(interface, duration)

    if packets.empty:
        print("No packets captured.")
        return

    features = aggregate_features(packets)

    sample = pd.DataFrame([features])

    for col in feature_columns:
        if col not in sample.columns:
            sample[col] = 0.0
    sample = sample[feature_columns]
    print("[i] Predicting...")
    prediction = rf_classifier.predict(sample)
    print(f"Captured {len(packets)} packets over {duration}s")
    print(f"Prediction: {'Attack' if prediction[0] == 1 else 'Benign'}")

feature_columns = X_train.columns.tolist()
predict_traffic(rf_classifier, feature_columns, interface='Wi-Fi', duration=10)

In [ ]:
import subprocess

result = subprocess.run(
    [r'C:\Program Files\Wireshark\tshark.exe', '-D'],
    capture_output=True, text=True
)
print(result.stdout)